In [16]:
import json

def count_ids(node):
    if isinstance(node, dict):
        return (1 if "id" in node else 0) + sum(count_ids(v) for v in node.values())
    if isinstance(node, list):
        return sum(count_ids(item) for item in node)
    return 0

with open("../dataset/Crop_Disease_train_qwenvl.json", "r") as f:
    data = json.load(f)
    print(count_ids(data))


145261


In [17]:
def make_batch_prompt(strands):
    strands_json = json.dumps(strands, ensure_ascii=False, indent=2)
    return f"""
    # CRITICAL KNOWLEDGE GRAPH EXTRACTION TASK

    You are a **Tier-1 Expert Knowledge Graph Architect and Data Extraction Engine**. Your specialization is the rapid, flawless transformation of raw agronomy data into a structured knowledge graph.

    Your task is to process the JSON array of *multiple, diverse conversations* regarding crop diseases and extract **ALL factual, non-redundant information** into a *single, flat, deduplicated list* of canonical triplets.

    ### STRICT ONTOLOGY & SCHEMA (NO DEVIATIONS ALLOWED)

    **NODE CATEGORIES (Subjects/Objects - ONLY THESE ARE PERMITTED):**
    * `Crop` (e.g., Tomato, Wheat, Grapes)
    * `Disease` (e.g., Fusarium Wilt, Leaf Spot)
    * `Symptom` (e.g., Yellowing leaves, Sunken lesions)
    * `Pathogen` (e.g., *Phytophthora infestans*, Virus)
    * `Treatment` (e.g., Fungicide, Crop rotation, Resistant variety)

    **RELATIONSHIP CATEGORIES (Predicates - ONLY THESE ARE PERMITTED):**
    * `AFFECTS`
    * `PRESENTS`
    * `CAUSED_BY`
    * `TREATED_BY`

    ### MANDATORY DIRECTIONALITY & CONSTRAINT ENFORCEMENT

    **YOU MUST ADHERE TO THE FOLLOWING DIRECTIONAL RULES. REVERSALS ARE A FATAL ERROR.**

    1.  **[Disease] $\rightarrow$ AFFECTS $\rightarrow$ [Crop]** (e.g., "Late Blight" AFFECTS "Potato")
    2.  **[Disease] $\rightarrow$ PRESENTS $\rightarrow$ [Symptom]** (e.g., "Downy Mildew" PRESENTS "Cottony growth")
    3.  **[Disease] $\rightarrow$ CAUSED_BY $\rightarrow$ [Pathogen]** (e.g., "Rust" CAUSED_BY "Fungus")
    4.  **[Disease] $\rightarrow$ TREATED_BY $\rightarrow$ [Treatment]** (e.g., "Clubroot" TREATED_BY "Lime application")

    **NEGATIVE CONSTRAINTS: DO NOT GENERATE THE FOLLOWING:**
    * **Node Types:** Do not use adjectives, locations, dates, or non-agronomy concepts as nodes.
    * **Redundancy:** Triplet list *must be fully unique*.
    * **Relationship Reversal:** e.g., **DO NOT** use `[Crop, AFFECTED_BY, Disease]`.
    * **Missing Labels:** All subject and object nodes must be clean, proper nouns (e.g., "Tomato" not "tomato plant").

    ### OUTPUT SPECIFICATION (JSON ONLY)

    **CRITICAL REQUIREMENT:** The final output **MUST BE** a single, valid JSON array of triplets.
    * **Format:** `[[subject, relationship, object], [subject_2, relationship_2, object_2], ...]`

    **Example of CORRECT, CANONICAL Output:**
    ```json
    [
      ["Powdery Mildew", "AFFECTS", "Cucumber"],
      ["Fusarium Wilt", "CAUSED_BY", "Fungus"],
      ["Fusarium Wilt", "PRESENTS", "Vascular discoloration"],
      ["Fusarium Wilt", "TREATED_BY", "Soil solarization"]
    ]
    ```
    
    Here is the batch of conversations to process:
    {strands_json}
    """

In [ ]:
import os
import json
import time
import random
from google import genai
from google.genai import types
from google.genai.errors import APIError
from tqdm import tqdm
from itertools import islice

INPUT_JSON = "../dataset/Crop_Disease_train_qwenvl.json"
OUTPUT_JSON = "../dataset/cddm_knowledge_graph_triplets.json"
PROCESSED_IDS_FILE = "../dataset/processed_strand_ids.json" 

BATCH_SIZE = 20
MODEL_NAME = "gemini-2.0-flash"

client = genai.Client()
print(f"Gemini client initialized with model: {MODEL_NAME}")


# --- GEMINI PARSING FUNCTION (Unchanged) ---
def parse_batch_with_gemini(strands):
    if not client:
        print("ERROR: Gemini client not initialized")
        return None

    prompt = make_batch_prompt(strands)

    triplet_list_schema = types.Schema(
        type=types.Type.ARRAY,
        items=types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
            min_items=3,
            max_items=3
        )
    )

    config = types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=triplet_list_schema,
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[prompt],
            config=config,
        )
        if response.text:
            return json.loads(response.text)
        else:
            print("⚠️ Empty response from API.")
            return None

    except APIError as e:
        print(f"API Error: {e}")
        return None
    except json.JSONDecodeError as e:
        raw_response_part = getattr(response, 'text', 'N/A')
        print(f"JSON Decode Error: {e}")
        print(f"Raw response part: {raw_response_part[:200]}...")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None


def safe_parse_batch_with_retry(strands, max_retries=5):
    for attempt in range(max_retries):
        results = parse_batch_with_gemini(strands)
        if results is None and attempt < max_retries - 1:
            wait_time = min(60, (2 ** attempt) + random.uniform(0, 2))
            print(f"⏳ Retrying in {wait_time:.1f}s (attempt #{attempt + 1})...")
            time.sleep(wait_time)
            continue
        return results if results is not None else []
    print("Max retries reached for batch.")
    return []

def load_existing_triplets():
    if os.path.exists(OUTPUT_JSON):
        try:
            with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
                triplets = json.load(f)
            print(f"Loaded {len(triplets)} previously saved triplets.")
            return {tuple(t) for t in triplets}
        except Exception as e:
            print(f"Could not load previous triplet file: {e}")
    return set()

def load_processed_ids():
    if os.path.exists(PROCESSED_IDS_FILE):
        try:
            with open(PROCESSED_IDS_FILE, "r", encoding="utf-8") as f:
                ids = set(json.load(f))
            print(f"Loaded {len(ids)} previously processed strand IDs.")
            return ids
        except Exception as e:
            print(f"Could not load processed IDs file: {e}")
    return set()

def save_triplets_atomic(triplet_set):
    tmp_file = OUTPUT_JSON + ".tmp"
    with open(tmp_file, "w", encoding="utf-8") as f:
        json.dump([list(t) for t in sorted(triplet_set)], f, ensure_ascii=False, indent=2)
    os.replace(tmp_file, OUTPUT_JSON)

def save_processed_ids_atomic(processed_ids):
    tmp_file = PROCESSED_IDS_FILE + ".tmp"
    with open(tmp_file, "w", encoding="utf-8") as f:
        json.dump(sorted(list(processed_ids)), f, ensure_ascii=False, indent=2)
    os.replace(tmp_file, PROCESSED_IDS_FILE)


with open(INPUT_JSON, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

for i, strand in enumerate(raw_data):
    strand['id'] = i

all_extracted_triplets = load_existing_triplets()
processed_strand_ids = load_processed_ids()

data_to_process = [
    strand for strand in raw_data
    if strand['id'] not in processed_strand_ids
]

total_strands = len(raw_data)
strands_to_process = len(data_to_process)
already_processed_count = total_strands - strands_to_process

print(f"\nTotal strands in input: {total_strands}")
print(f"Strands already processed: {already_processed_count} (SKIPPING)")
print(f"Strands remaining to process: {strands_to_process}")
print(f"--- Starting Processing ---\n")

data_iterator = iter(data_to_process)

with tqdm(total=strands_to_process, desc="Extracting KG Triplets") as pbar:
    for batch in iter(lambda: list(islice(data_iterator, BATCH_SIZE)), []):
        
        triplets = safe_parse_batch_with_retry(batch)
        
        batch_ids = [strand['id'] for strand in batch]

        if triplets:
            for triplet in triplets:
                all_extracted_triplets.add(tuple(triplet))

            processed_strand_ids.update(batch_ids)
            
            save_triplets_atomic(all_extracted_triplets)
            save_processed_ids_atomic(processed_strand_ids)

        else:
            print(f"Batch failed completely. IDs not marked as processed: {batch_ids}")
        
        pbar.update(len(batch))
        time.sleep(1.0) 

print("\n--- Summary ---")
print(f"Total strands in input: {total_strands}")
print(f"Total strands marked as processed: {len(processed_strand_ids)}")
print(f"Total UNIQUE triplets found: {len(all_extracted_triplets)}")
print(f"Triplets saved to {OUTPUT_JSON}")
print(f"Processed IDs saved to {PROCESSED_IDS_FILE}")

In [ ]:
import json
from tqdm import tqdm
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv
load_dotenv()

JSON_PATH = "../dataset/cddm_knowledge_graph_triplets.json"   

driver = GraphDatabase.driver(os.getenv("NEO4J_URI"), auth=(os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD")))

with open(JSON_PATH, "r") as f:
    data = json.load(f)

with driver.session() as session:
    for source, relation, target in tqdm(data, desc="Creating graph", unit="rel"):
        relation = relation.upper().replace(" ", "_")
        query = f"""
        MERGE (a:Entity {{name: $source}})
        MERGE (b:Entity {{name: $target}})
        MERGE (a)-[:{relation}]->(b)
        """
        session.run(query, source=source, target=target)

print("Graph built!")
driver.close()